Splitting labels into two groups fake news and reliable.

In [1]:
import pandas as pd
import ast
from rich.progress import track
from sklearn.feature_extraction.text import CountVectorizer
from sklearn import linear_model
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import swifter
import numpy as np

In [3]:
all_columns = [
    'id', 'domain', 'type', 'url', 'content', 'scraped_at', 'inserted_at', 
    'updated_at', 'title', 'authors', 'keywords', 'meta_keywords', 
    'meta_description', 'tags', 'summary']
converters_dict = {col: ast.literal_eval for col in all_columns}

training_set = pd.read_csv('../Group_project/training_set.csv', converters = converters_dict)

In [4]:
# creating list of 10000 most common words from vocabulary found in part 1.
full_voc = pd.read_csv('../Group_project/vocabulary_995000.csv')
sub_voc = set((full_voc.iloc[:10000])['word'].values.tolist())
print(sub_voc)

{'refut', 'collect', 'scrub', 'mae', 'upbeat', 'nutrit', 'worldwid', 'northwestern', 'ag', 'passov', 'charter', 'tuck', 'sooner', 'marina', 'jen', 'admit', 'chronic', 'vehicl', 'chord', 'bronco', 'defin', 'von', 'xl', 'financi', 'compulsori', 'reckon', 'equit', 'rachael', 'trumprussia', 'welfar', 'elder', 'doubl', 'underw', 'novosti', 'calcul', 'greek', 'squirrel', 'misinform', 'cholesterol', 'unchang', 'dinner', 'plug', 'derek', 'laughter', 'fraud', 'grave', 'chapel', 'reinforc', 'think', 'cod', 'product', 'hammond', 'secondari', 'proper', 'tomb', 'illeg', 'cat', 'guggenheim', 'algorithm', 'verizon', 'baton', 'testosteron', 'por', 'brock', 'reconnaiss', 'tumblr', 'actual', 'sector', 'miniatur', 'marxism', 'waiter', 'sausag', 'garag', 'misguid', 'gigant', 'con', 'infecti', 'mold', 'formal', 'expir', 'time', 'technician', 'ivi', 'drank', 'behead', 'femal', 'share', 'lit', 'arlen', 'donald', 'retire', 'undercut', 'array', 'adopt', 'loss', 'skill', 'congressmen', 'ryan', 'new', 'angola', 

In [5]:
types = {i[0] for i in training_set['type'] if len(i) < 2}
print(types)
fake_news = {'fake', 'satir', 'bias', 'conspiraci', 'junksci', 'hate', 'rumor'}
reliable_news = {'reliabl', 'unreli', 'polit', 'clickbait'}

fake_only_rel = {'clickbait', 'unreli', 'polit', 'fake', 'satir', 'bias', 'conspiraci', 'junksci', 'hate', 'rumor'}
rel_only_rel = {'reliabl'}

def filter_type(type, fake, rel):
    if type in fake:
        return 'fake'
    elif type in rel:
        return 'reliable'

{'hate', 'satir', 'junksci', 'unreli', 'reliabl', 'clickbait', 'polit', 'fake', 'bias', 'unknown', 'conspiraci', 'rumor'}


In [23]:
def filter_strings(str_list, **kwargs):
    filtered = [word for word in str_list if word in sub_voc]
    joined = ' '.join(filtered)
    return joined

# this is the model with even type dist and only content col
only_content = pd.DataFrame({})
only_content['content'] = training_set['content'].swifter.progress_bar(False).apply(filter_strings)
only_content['y_t'] = training_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_news, reliable_news))

# this is the model with only reliable as reliable type and only content col
only_content_rel = pd.DataFrame({})
only_content_rel['content'] = training_set['content'].swifter.progress_bar(False).apply(filter_strings)
only_content_rel['y_t'] = training_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_only_rel, rel_only_rel))

relevant_columns = [
    'content', 'title', 'authors', 'keywords', 'meta_keywords', 
    'meta_description', 'tags', 'summary']

ad_domains = [
    'content', 'title', 'authors', 'keywords', 'meta_keywords', 
    'meta_description', 'tags', 'summary', 'domain']

def filter_df(df, cols):
    f_df = pd.DataFrame({})
    for i in track(cols, 'processing...'):
        f_df[i] = df[i].swifter.progress_bar(False).apply(filter_strings)
    return f_df

# this is the model with even type dist and all relevant cols
filtered = filter_df(training_set, relevant_columns)
filtered['y_t'] = training_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_news, reliable_news))

# this is the model with only reliable as reliable type and all relevant cols
filtered_rel = filter_df(training_set, relevant_columns)
filtered_rel['y_t'] = training_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_only_rel, rel_only_rel))

# this is the model with all relevant columns and domains and even dist of types 
filtered_dom = filter_df(training_set, relevant_columns)
filtered_dom['domain'] = training_set['domain'].swifter.progress_bar(False).apply(lambda x, **kwargs: ' '.join(x))
filtered_dom['y_t'] = training_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_news, reliable_news))

Output()

Output()

Output()

In [25]:
# this is the model with even type dist and only content col
x_oc = only_content[only_content['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
y_oc = x_oc['y_t']

# this is the model with only reliable as reliable type and only content col
x_oc_rel = only_content_rel[only_content_rel['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
y_oc_rel = x_oc_rel['y_t']

# this is the model with even type dist and all relevant cols
x_t = filtered[filtered['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
y_t = x_t['y_t']

# this is the model with only reliable as reliable type and all relevant cols
x_t_rel = filtered_rel[filtered_rel['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
y_t_rel = x_t_rel['y_t']

# this is the model with all relevant columns and domains and even dist of types 
x_t_dom = filtered_dom[filtered_dom['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
y_t_dom = x_t_dom['y_t']

# this is the model with even type dist and only content col
vectorizer_oc = CountVectorizer()
X_oc = vectorizer_oc.fit_transform(x_oc['content'])
Y_oc = np.array(y_oc)

# this is the model with only reliable as reliable type and only content col
vectorizer_oc_rel = CountVectorizer()
X_oc_rel = vectorizer_oc_rel.fit_transform(x_oc_rel['content'])
Y_oc_rel = np.array(y_oc_rel)

transformer_list = [(f'vec_{col}', CountVectorizer(), col) for col in relevant_columns]

# this is the model with even type dist and all relevant cols
transformer = ColumnTransformer(transformers=transformer_list, remainder='drop')
X_train = transformer.fit_transform(x_t)
Y_train = np.array(y_t)

# this is the model with only reliable as reliable type and all relevant cols
transformer_rel = ColumnTransformer(transformers=transformer_list, remainder='drop')
X_train_rel = transformer_rel.fit_transform(x_t_rel)
Y_train_rel = np.array(y_t_rel)

# this is the model with all relevant columns and domains and even dist of types 
transformer_list_dom = [(f'vec_{col}', CountVectorizer(), col) for col in ad_domains]
transformer_dom = ColumnTransformer(transformers=transformer_list_dom, remainder='drop')
X_train_dom = transformer_dom.fit_transform(x_t_dom)
Y_train_dom = np.array(y_t_dom)

In [43]:
# this is the model with even type dist and only content col
logr = linear_model.LogisticRegression(penalty = 'l2', max_iter = 100000)
logr.fit(X_oc, Y_oc)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

In [ ]:
# this is the model with only reliable as reliable type and only content col
logr_rel_only_rel = linear_model.LogisticRegression(penalty = 'l2', max_iter = 100000)
logr_rel_only_rel.fit(X_oc_rel, Y_oc_rel)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

In [ ]:
# this is the model with even type dist and all relevant cols
logr_2_3 = linear_model.LogisticRegression(penalty = 'l2', max_iter = 100000)
logr_2_3.fit(X_train, Y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

In [ ]:
# this is the model with only reliable as reliable type and all relevant cols
logr_2_3_rel_only_rel = linear_model.LogisticRegression(penalty = 'l2', max_iter = 100000)
logr_2_3_rel_only_rel.fit(X_train_rel, Y_train_rel)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

In [44]:
# this is the model with all relevant columns and domains and even dist of types 
logr_2_3_dom = linear_model.LogisticRegression(penalty = 'l2', max_iter = 100000)
logr_2_3_dom.fit(X_train_dom, Y_train_dom)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

In [21]:
test_set = pd.read_csv('../Group_project/test_set.csv', converters = converters_dict)

In [45]:
# this is the model with even type dist and only content col
oc_test = pd.DataFrame({})
oc_test['content'] = test_set['content'].swifter.progress_bar(False).apply(filter_strings)
oc_test['y_t'] = test_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_news, reliable_news))

# this is the model with only reliable as reliable type and only content col
oc_test_rel = pd.DataFrame({})
oc_test_rel['content'] = test_set['content'].swifter.progress_bar(False).apply(filter_strings)
oc_test_rel['y_t'] = test_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_only_rel, rel_only_rel))

# this is the model with even type dist and all relevant cols
filtered_test = filter_df(test_set, relevant_columns)
filtered_test['y_t'] = test_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_news, reliable_news))

# this is the model with only reliable as reliable type and all relevant cols
filtered_test_rel = filter_df(test_set, relevant_columns)
filtered_test_rel['y_t'] = test_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_only_rel, rel_only_rel))

# this is the model with all relevant columns and domains and even dist of types 
filtered_test_dom = filter_df(test_set, relevant_columns)
filtered_test_dom['domain'] = test_set['domain'].swifter.progress_bar(False).apply(lambda x, **kwargs: ' '.join(x))
filtered_test_dom['y_t'] = test_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_news, reliable_news))

Output()

Output()

Output()

In [46]:
# this is the model with even type dist and only content col
x_oc_t = oc_test[oc_test['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
oc_y = x_oc_t['y_t']

# this is the model with only reliable as reliable type and only content col
x_oc_t_rel = oc_test_rel[oc_test_rel['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
oc_y_rel = x_oc_t_rel['y_t']

# this is the model with even type dist and only content col
OC_test = vectorizer_oc.transform(x_oc_t['content'])
oc_Y_test = np.array(oc_y)

# this is the model with only reliable as reliable type and only content col
OC_test_rel = vectorizer_oc_rel.transform(x_oc_t_rel['content'])
oc_Y_test_rel = np.array(oc_y_rel)

# this is the model with even type dist and all relevant cols
x_test = filtered_test[filtered_test['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
yt_t = x_test['y_t']

# this is the model with only reliable as reliable type and all relevant cols
x_test_rel = filtered_test_rel[filtered_test_rel['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
yt_t_rel = x_test_rel['y_t']

# this is the model with all relevant columns and domains and even dist of types 
x_test_dom = filtered_test_dom[filtered_test_dom['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
yt_t_dom = x_test_dom['y_t']

# this is the model with even type dist and all relevant cols
X_test = transformer.transform(x_test)
y_test = np.array(yt_t)

# this is the model with only reliable as reliable type and all relevant cols
X_test_rel = transformer_rel.transform(x_test_rel)
y_test_rel = np.array(yt_t_rel)

# this is the model with all relevant columns and domains and even dist of types 
X_test_dom = transformer_dom.transform(x_test_dom)
y_test_dom = np.array(yt_t_dom)

In [ ]:
# this is the model with even type dist and only content col
predictions = logr.predict(OC_test)

# this is the model with only reliable as reliable type and only content col
predictions_rel_only_rel = logr_rel_only_rel.predict(OC_test_rel)

# this is the model with even type dist and all relevant cols
predictions_2_3 = logr_2_3.predict(X_test)

# this is the model with only reliable as reliable type and all relevant cols
predictions_2_3_rel_only_rel = logr_2_3_rel_only_rel.predict(X_test_rel)

# this is the model with all relevant columns and domains and even dist of type
predictions_2_3_dom = logr_2_3_dom.predict(X_test_dom)

In [44]:
print('The scores for model where only column "content" is used and with an even distribution of the types')
print(f'The acuracy score for the model is: {accuracy_score(oc_Y_test, predictions)}')
print('The confusion matrix for the model:')
print(confusion_matrix(oc_Y_test, predictions))
print('The classification report of the logistic model is:')
print(classification_report(oc_Y_test, predictions))

The scores for model where only column "content" is used and with an even distribution of the types
The acuracy score for the model is: 0.8493961218836565
The confusion matrix for the model:
[[37706  5035]
 [ 8557 38952]]
The classification report of the logistic model is:
              precision    recall  f1-score   support

        fake       0.82      0.88      0.85     42741
    reliable       0.89      0.82      0.85     47509

    accuracy                           0.85     90250
   macro avg       0.85      0.85      0.85     90250
weighted avg       0.85      0.85      0.85     90250



In [45]:
print('The scores for model where only column "content" is used and where reliable type is only type reliable and all other types are fake')
print(f'The acuracy score for the model is: {accuracy_score(oc_Y_test_rel, predictions_rel_only_rel)}')
print('The confusion matrix for the model:')
print(confusion_matrix(oc_Y_test_rel, predictions_rel_only_rel))
print('The classification report of the logistic model is:')
print(classification_report(oc_Y_test_rel, predictions_rel_only_rel))

The scores for model where only column "content" is used and where reliable type is only type reliable and all other types are fake
The acuracy score for the model is: 0.9348365650969529
The confusion matrix for the model:
[[66873  1691]
 [ 4190 17496]]
The classification report of the logistic model is:
              precision    recall  f1-score   support

        fake       0.94      0.98      0.96     68564
    reliable       0.91      0.81      0.86     21686

    accuracy                           0.93     90250
   macro avg       0.93      0.89      0.91     90250
weighted avg       0.93      0.93      0.93     90250



In [46]:
print('The scores for model where all relevant columns are used and where types are evenly distributed')
print(f'The acuracy score for the model is: {accuracy_score(y_test, predictions_2_3)}')
print('The confusion matrix for the model:')
print(confusion_matrix(y_test, predictions_2_3))
print('The classification report of the logistic model is:')
print(classification_report(y_test, predictions_2_3))

The scores for model where all relevant columns are used and where types are evenly distributed
The acuracy score for the model is: 0.9594238227146814
The confusion matrix for the model:
[[41295  1446]
 [ 2216 45293]]
The classification report of the logistic model is:
              precision    recall  f1-score   support

        fake       0.95      0.97      0.96     42741
    reliable       0.97      0.95      0.96     47509

    accuracy                           0.96     90250
   macro avg       0.96      0.96      0.96     90250
weighted avg       0.96      0.96      0.96     90250



In [ ]:
print('The scores for model where all relevant columns are used and where reliable type is only type reliable and all other types are fake')
print(f'The acuracy score for the model is: {accuracy_score(y_test_rel, predictions_2_3_rel_only_rel)}')
print('The confusion matrix for the model:')
print(confusion_matrix(y_test_rel, predictions_2_3_rel_only_rel))
print('The classification report of the logistic model is:')
print(classification_report(y_test_rel, predictions_2_3_rel_only_rel))

The scores for model where only column "content" is used and where reliable type is only type reliable and all other types are fake
The acuracy score for the model is: 0.9874792243767313
The confusion matrix for the model:
[[68066   498]
 [  632 21054]]
The classification report of the logistic model is:
              precision    recall  f1-score   support

        fake       0.99      0.99      0.99     68564
    reliable       0.98      0.97      0.97     21686

    accuracy                           0.99     90250
   macro avg       0.98      0.98      0.98     90250
weighted avg       0.99      0.99      0.99     90250



In [48]:
print('The scores for model where all relevant columns and domains are used and where types are evenly distributed')
print(f'The acuracy score for the model is: {accuracy_score(y_test_dom, predictions_2_3_dom)}')
print('The confusion matrix for the model:')
print(confusion_matrix(y_test_dom, predictions_2_3_dom))
print('The classification report of the logistic model is:')
print(classification_report(y_test_dom, predictions_2_3_dom))

The scores for model where all relevant columns and domains are used and where types are evenly distributed
The acuracy score for the model is: 0.9983601108033241
The confusion matrix for the model:
[[42683    58]
 [   90 47419]]
The classification report of the logistic model is:
              precision    recall  f1-score   support

        fake       1.00      1.00      1.00     42741
    reliable       1.00      1.00      1.00     47509

    accuracy                           1.00     90250
   macro avg       1.00      1.00      1.00     90250
weighted avg       1.00      1.00      1.00     90250

